# Harsh's Blueprint: Feature Engineering, CART Baseline & Hyperparameter Tuning
## DNS Tunneling Detection ML Pipeline — Stages 7, 8 & 10 (Part 1)

**Role:** Feature Architect & Baseline Modeler (Harsh)
**Upstream:** Krishna (Stages 1–6: Data Ingestion, Cleaning, OHE, Scaling & SMOTE)
**Downstream:** Jaimin (Stages 9, 10 Part 2, 11–12: Deep Learning & SVMs)

---

### Assigned Objectives

| Stage | Title | Deliverable |
|-------|-------|-------------|
| **7** | Feature Engineering & Selection | Engineered feature matrix with Shannon entropy proxies, subdomain length signals, record-type volatility, and multicollinearity reduction |
| **8** | Baseline Model (CART Decision Tree) | Trained, interpretable Decision Tree with visual rule diagram and confusion matrix |
| **10 (Part 1)** | Hyperparameter Tuning | 5-Fold Stratified `GridSearchCV` optimising `max_depth`, `min_samples_split`, `min_samples_leaf`, and `criterion` |

### Pipeline Position
$$\underbrace{X_{\text{train\_bal}}, y_{\text{train\_bal}}, X_{\text{test}}, y_{\text{test}}}_{\text{Krishna's Deliverables (Scaled, Balanced)}} \longrightarrow \underbrace{\text{Feature Engineering} \to \text{CART Baseline} \to \text{GridSearchCV}}_{\text{Harsh's Pipeline}} \longrightarrow \underbrace{\text{SVM, Neural Networks}}_{\text{Jaimin}}$$


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.size': 11, 'figure.titlesize': 14, 'axes.labelsize': 12})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DIR = Path('data/processed')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")


---
## Loading Krishna's Deliverables (Stages 1–6 Outputs)

We ingest the **pre-processed, scaled, and SMOTE-balanced** datasets produced by Krishna's pipeline:
- `X_train_bal.csv` — $158{,}598 \times 28$ scaled balanced training features
- `y_train_bal.csv` — $158{,}598$ balanced labels (50/50)
- `X_test.csv` — $34{,}071 \times 28$ scaled pristine test features
- `y_test.csv` — $34{,}071$ labels (authentic 0.25% attack rate)


In [ ]:
# Load Krishna's deliverables
X_train = pd.read_csv(PROCESSED_DIR / 'X_train_bal.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train_bal.csv').squeeze()
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

print("=== Krishna's Deliverables Loaded ===")
print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}  |  Balance: {dict(y_train.value_counts())}")
print(f"X_test:  {X_test.shape}  |  y_test:  {y_test.shape}  |  Balance: {dict(y_test.value_counts())}")
print(f"\nFeatures ({X_train.shape[1]}): {X_train.columns.tolist()}")
print(f"\nNull check — Train: {X_train.isnull().sum().sum()}, Test: {X_test.isnull().sum().sum()}")


---
## Stage 7: Feature Engineering & Selection

### Objective
Extract and construct mathematical features and signals that specifically expose DNS tunneling. We operate on Krishna's **cleaned, scaled, and SMOTE-balanced** feature matrix. Our feature engineering directly implements the three blueprint features:

#### Feature 1: Shannon Entropy
* **Concept**: Measures the randomness of a string. Normal domains (`google.com`) adhere to standard English phonetics (low entropy). Tunneling domains (`a1b2c3d4.evil.com`) contain Base64/Base32-encoded encrypted payloads, appearing completely random (high entropy).
* **Mathematical Formula**:
  $$H(X) = -\sum_{i=1}^{n} p(x_i) \log_2 p(x_i)$$
  where $p(x_i)$ represents the empirical relative frequency of character $x_i$.
* **Derived Interaction**: `entropy_x_length = domain_entropy * query_length` captures queries exhibiting *both* long length and high entropy.

#### Feature 2: Subdomain Length & Count
* **Concept**: DNS (RFC 1035) restricts total query length to 255 characters, and each subdomain label between dots (`.`) to 63 characters. Attackers pack exactly 63 random characters into multiple subdomains to exfiltrate the maximum payload per packet.
* **Signals**:
  - `query_length` captures the total query length (bounded up to 255 chars).
  - Dot counts (`.`) and subdomain length boundaries are reflected through `query_length` and label aggregations.

#### Feature 3: Record Type Volatility
* **Concept**: Attackers prefer using `TXT` and `NULL` records because they permit significantly larger payload response buffers from the C2 server compared to standard `A` records.
* **Action**: Ensure Krishna's One-Hot Encoding of these record types (`rr_type_TXT`, `rr_type_NONE`, `rr_type_A`) is fully leveraged.
* **Derived Score**: `tunneling_rr_score = rr_type_TXT + rr_type_NONE - rr_type_A` weights tunneling-exploited record types positively and standard `A` lookups negatively.

#### Composite Network Diversity Signal
* **Concept**: Tunneling traffic repeatedly queries the adversary's single authoritative nameserver (C2), whereas benign enterprise traffic connects to diverse CDNs, autonomous systems, and nations.
* `network_diversity = unique_country_count + unique_asn_count + distinct_domains_count`

### Feature Selection: Multicollinearity & Zero-Variance Reduction
EDA (Stage 6) identified extreme collinearity ($|r| > 0.90$) and constant zero-variance features. We drop redundant columns to eliminate variance inflation and ensure clear, interpretable tree splits.


In [ ]:
# =====================================================================
# STEP 7.1: Engineer New Interaction Features
# =====================================================================

def engineer_features(df):
    # Apply feature engineering transforms to a feature dataframe.
    df = df.copy()

    # Signal 1: Entropy × Length interaction
    df['entropy_x_length'] = df['domain_entropy'] * df['query_length']

    # Signal 2: Tunneling record-type volatility score
    df['tunneling_rr_score'] = df['rr_type_TXT'] + df['rr_type_NONE'] - df['rr_type_A']

    # Signal 3: Network diversity composite
    df['network_diversity'] = (df['unique_country_count'] +
                               df['unique_asn_count'] +
                               df['distinct_domains_count'])
    return df

X_train_eng = engineer_features(X_train)
X_test_eng  = engineer_features(X_test)

print("=== Engineered Features Added ===")
new_feats = ['entropy_x_length', 'tunneling_rr_score', 'network_diversity']
print(f"New features: {new_feats}")
print(f"X_train shape: {X_train.shape} -> {X_train_eng.shape}")
print(f"X_test shape:  {X_test.shape} -> {X_test_eng.shape}")

# Quick statistical profile of new features
print("\nTrain statistics for engineered features (by class):")
for feat in new_feats:
    b_mean = X_train_eng.loc[y_train == 0, feat].mean()
    a_mean = X_train_eng.loc[y_train == 1, feat].mean()
    print(f"  {feat}: Benign mean={b_mean:.4f}, Attack mean={a_mean:.4f}, Separation={abs(a_mean-b_mean):.4f}")


In [ ]:
# =====================================================================
# STEP 7.2: Feature Selection — Remove Zero-Variance & Collinear
# =====================================================================

# Identify zero/near-zero variance columns
zero_var_cols = [c for c in X_train_eng.columns if X_train_eng[c].std() < 0.001]
print(f"Zero/near-zero variance features to DROP ({len(zero_var_cols)}):")
for c in zero_var_cols:
    print(f"  - {c} (std={X_train_eng[c].std():.6f}, unique={X_train_eng[c].nunique()})")

# Drop zero-variance features
X_train_sel = X_train_eng.drop(columns=zero_var_cols)
X_test_sel  = X_test_eng.drop(columns=zero_var_cols)

# Identify and resolve high multicollinearity (|r| > 0.95)
# Based on EDA findings, drop the less informative member of each pair:
# - unique_asn_count (r=0.9996 with unique_country_count) -> DROP
# - distinct_domains_count (r=0.989 with unique_country_count) -> DROP
# - TXT_frequency (r=1.00 with rr_type_TXT) -> DROP (keep the OHE version)
# - query_length (r=0.98 with domain_entropy) -> KEEP BOTH (key features, tree can handle)

collinear_drops = []
drop_candidates = {
    'unique_asn_count': 'Perfectly correlated (r=0.9996) with unique_country_count',
    'distinct_domains_count': 'Highly correlated (r=0.989) with unique_country_count',
    'TXT_frequency': 'Perfectly correlated (r=1.00) with rr_type_TXT',
}

for col, reason in drop_candidates.items():
    if col in X_train_sel.columns:
        collinear_drops.append(col)
        print(f"\nDROPPING collinear feature: '{col}'")
        print(f"  Reason: {reason}")

X_train_sel = X_train_sel.drop(columns=collinear_drops, errors='ignore')
X_test_sel  = X_test_sel.drop(columns=collinear_drops, errors='ignore')

print(f"\n=== Feature Selection Complete ===")
print(f"Original features: {X_train_eng.shape[1]}")
print(f"Dropped (zero-var): {len(zero_var_cols)}")
print(f"Dropped (collinear): {len(collinear_drops)}")
print(f"Final features: {X_train_sel.shape[1]}")
print(f"\nFinal feature set ({X_train_sel.shape[1]}):")
for i, c in enumerate(X_train_sel.columns, 1):
    print(f"  {i:2d}. {c}")


### Feature Importance Analysis via Mutual Information

Before training any model, we assess each feature's **univariate predictive power** using **Mutual Information** (MI):

$$I(X; Y) = \sum_{y \in Y} \sum_{x \in X} p(x, y) \log_2 \frac{p(x, y)}{p(x) \cdot p(y)}$$

MI quantifies the reduction in uncertainty about the target class $Y$ given knowledge of feature $X$. Unlike Pearson correlation, MI captures **non-linear dependencies** — essential because DNS tunneling features often exhibit threshold-based (step-function) relationships with the target.


In [ ]:
# Mutual Information scoring
mi_scores = mutual_info_classif(X_train_sel, y_train, random_state=RANDOM_STATE, n_neighbors=5)
mi_df = pd.DataFrame({
    'Feature': X_train_sel.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False).reset_index(drop=True)

print("=== Mutual Information Scores (Univariate Predictive Power) ===")
print(mi_df.to_string(index=False))

# Visualise MI scores
plt.figure(figsize=(10, 6))
colors = ['#d62728' if s > mi_df['MI_Score'].median() else '#1f77b4' for s in mi_df['MI_Score']]
plt.barh(mi_df['Feature'], mi_df['MI_Score'], color=colors)
plt.xlabel('Mutual Information Score')
plt.title('Stage 7: Feature Importance — Mutual Information Scores', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.axvline(x=mi_df['MI_Score'].median(), color='black', linestyle='--', alpha=0.5, label='Median MI')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage7_mutual_information_scores.png', dpi=300)
plt.show()
print(f"\nSaved: {FIGURES_DIR / 'stage7_mutual_information_scores.png'}")


In [ ]:
# Visualise engineered feature distributions by class
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

eng_feats = ['entropy_x_length', 'tunneling_rr_score', 'network_diversity']
titles = [
    'Entropy × Length\n(High = likely tunneling)',
    'Tunneling RR Score\n(Positive = TXT/NULL dominant)',
    'Network Diversity\n(Low = single C2 server)'
]

for ax, feat, title in zip(axes, eng_feats, titles):
    for label, color, name in [(0, '#1f77b4', 'Benign'), (1, '#d62728', 'Attack')]:
        data = X_train_sel.loc[y_train == label, feat]
        ax.hist(data, bins=50, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Stage 7: Engineered Feature Distributions by Class', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage7_engineered_features_distribution.png', dpi=300)
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage7_engineered_features_distribution.png'}")


---
## Stage 8: Baseline Model — CART Decision Tree

### Algorithm: Classification and Regression Trees (CART)
The Decision Tree Classifier recursively partitions the feature space into rectangular regions. At each internal node, the algorithm selects the feature and threshold that **maximises purity gain** in the child nodes.

### Splitting Criterion: Gini Impurity
At each node $t$, the **Gini Impurity** measures the probability that a randomly chosen sample would be misclassified if labelled according to the class distribution at that node:

$$\text{Gini}(t) = 1 - \sum_{i=1}^{C} (p_i)^2$$

where $p_i$ is the fraction of class $i$ samples at node $t$. A Gini index of $0$ indicates perfect node purity (all samples belong to one class).

### Why Decision Tree as Baseline?
1. **Interpretability**: Security Operations Center (SOC) analysts need **transparent, auditable rules** (e.g., `IF entropy > 3.2 AND query_length > 25 THEN Malicious`). Decision trees provide exactly this.
2. **No Feature Scaling Assumption**: Unlike SVMs or Neural Networks, CART is invariant to monotonic feature transformations — it only cares about rank ordering. This makes it a robust sanity check on the preprocessed data.
3. **Benchmark Principle**: Jaimin's complex models (Neural Networks, SVMs) must **exceed** this interpretable baseline to justify their computational cost and opacity.

### Initial Baseline: Default Hyperparameters
We first train an **unconstrained** tree (no depth limit) to establish the raw ceiling, then use Stage 10 tuning to find the optimal regularised configuration.


In [ ]:
# =====================================================================
# STEP 8.1: Train Default (Unconstrained) CART Decision Tree
# =====================================================================
dt_baseline = DecisionTreeClassifier(
    criterion='gini',
    random_state=RANDOM_STATE
)

dt_baseline.fit(X_train_sel, y_train)

# Predictions
y_pred_train = dt_baseline.predict(X_train_sel)
y_pred_test  = dt_baseline.predict(X_test_sel)

print("=" * 70)
print("  STAGE 8: CART DECISION TREE — DEFAULT (UNCONSTRAINED) BASELINE")
print("=" * 70)
print(f"\nTree Depth: {dt_baseline.get_depth()}")
print(f"Number of Leaves: {dt_baseline.get_n_leaves()}")
print(f"Number of Features Used: {dt_baseline.n_features_in_}")

print("\n--- Training Set Performance ---")
print(f"Accuracy:  {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Precision: {precision_score(y_train, y_pred_train):.4f}")
print(f"Recall:    {recall_score(y_train, y_pred_train):.4f}")
print(f"F1-Score:  {f1_score(y_train, y_pred_train):.4f}")

print("\n--- Test Set Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_test):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_test):.4f}")

print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_test, target_names=['Benign (0)', 'Malicious (1)']))


In [ ]:
# =====================================================================
# STEP 8.2: Confusion Matrix Visualization
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training confusion matrix
cm_train = confusion_matrix(y_train, y_pred_train)
ConfusionMatrixDisplay(cm_train, display_labels=['Benign', 'Attack']).plot(
    ax=axes[0], cmap='Blues', values_format=',d')
axes[0].set_title('Training Set Confusion Matrix', fontsize=13, fontweight='bold')

# Test confusion matrix
cm_test = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(cm_test, display_labels=['Benign', 'Attack']).plot(
    ax=axes[1], cmap='Reds', values_format=',d')
axes[1].set_title('Test Set Confusion Matrix', fontsize=13, fontweight='bold')

plt.suptitle('Stage 8: CART Baseline — Confusion Matrices (Default Hyperparameters)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage8_baseline_confusion_matrix.png', dpi=300)
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage8_baseline_confusion_matrix.png'}")


In [ ]:
# =====================================================================
# STEP 8.3: Decision Tree Feature Importance (Gini-based)
# =====================================================================
importances = dt_baseline.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': X_train_sel.columns,
    'Gini_Importance': importances
}).sort_values('Gini_Importance', ascending=False)

plt.figure(figsize=(10, 6))
top_n = min(15, len(feat_imp))
top_feats = feat_imp.head(top_n)
colors = ['#d62728' if f in ['entropy_x_length', 'tunneling_rr_score', 'network_diversity']
          else '#1f77b4' for f in top_feats['Feature']]
plt.barh(top_feats['Feature'], top_feats['Gini_Importance'], color=colors)
plt.xlabel('Gini Feature Importance')
plt.title('Stage 8: CART Baseline — Feature Importances (Top 15)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage8_baseline_feature_importance.png', dpi=300)
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage8_baseline_feature_importance.png'}")

print("\nFull Feature Importance Ranking:")
print(feat_imp.to_string(index=False))


In [ ]:
# =====================================================================
# STEP 8.4: Decision Tree Visualization (Compact — max_depth=4)
# =====================================================================
# Visualise the top 4 levels of the tree for interpretability
plt.figure(figsize=(24, 12))
plot_tree(
    dt_baseline,
    max_depth=4,
    feature_names=X_train_sel.columns.tolist(),
    class_names=['Benign', 'Attack'],
    filled=True,
    rounded=True,
    fontsize=8,
    proportion=True,
    impurity=True
)
plt.title('Stage 8: CART Decision Tree — Top 4 Levels (Visual Rule Diagram)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage8_decision_tree_visualization.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage8_decision_tree_visualization.png'}")

# Print top decision rules as text
print("\n=== Decision Rules (Top 6 Levels) ===")
tree_rules = export_text(dt_baseline, feature_names=X_train_sel.columns.tolist(), max_depth=6)
print(tree_rules[:3000])


### Overfitting Diagnostic: Default vs Constrained

The **unconstrained** tree ($\texttt{max\_depth}=\infty$) likely achieves **perfect training accuracy** but lower test performance — this gap is the hallmark of **overfitting**.

An unconstrained tree memorises training noise (including SMOTE-synthesised points), creating hyper-specific decision boundaries that fail to generalise. This motivates **Stage 10: Hyperparameter Tuning** to find the optimal complexity trade-off.


In [ ]:
# Overfitting diagnostic: train vs test across max_depth
depths = list(range(1, 31)) + [None]
train_scores = []
test_scores = []

for d in depths:
    dt_temp = DecisionTreeClassifier(criterion='gini', max_depth=d, random_state=RANDOM_STATE)
    dt_temp.fit(X_train_sel, y_train)
    train_scores.append(f1_score(y_train, dt_temp.predict(X_train_sel)))
    test_scores.append(f1_score(y_test, dt_temp.predict(X_test_sel)))

plt.figure(figsize=(12, 5))
x_labels = [str(d) if d is not None else 'None' for d in depths]
plt.plot(range(len(depths)), train_scores, 'b-o', markersize=3, label='Train F1', alpha=0.8)
plt.plot(range(len(depths)), test_scores, 'r-s', markersize=3, label='Test F1', alpha=0.8)
plt.fill_between(range(len(depths)), train_scores, test_scores, alpha=0.15, color='gray')

# Mark best test depth
best_idx = np.argmax(test_scores)
plt.axvline(x=best_idx, color='green', linestyle='--', alpha=0.7, label=f'Best Test F1 @ depth={depths[best_idx]}')
plt.scatter([best_idx], [test_scores[best_idx]], color='green', s=100, zorder=5)

plt.xticks(range(0, len(depths), 3), [x_labels[i] for i in range(0, len(depths), 3)], rotation=45)
plt.xlabel('max_depth')
plt.ylabel('F1 Score')
plt.title('Stage 8: Overfitting Diagnostic — Train vs Test F1 Across Tree Depths', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage8_overfitting_diagnostic.png', dpi=300)
plt.show()

print(f"Best test F1: {test_scores[best_idx]:.4f} at max_depth={depths[best_idx]}")
print(f"Train F1 gap at best depth: {train_scores[best_idx] - test_scores[best_idx]:.4f} (overfit indicator)")
print(f"Saved: {FIGURES_DIR / 'stage8_overfitting_diagnostic.png'}")


---
## Stage 10 (Part 1): Hyperparameter Tuning via 5-Fold GridSearchCV

### The Problem: Unconstrained Trees Overfit
An unconstrained CART tree grows until every leaf is pure (Gini = 0), effectively **memorising** the training set. The tree learns noise, SMOTE interpolation artifacts, and idiosyncratic patterns that do not generalise. This produces a large gap between training accuracy ($\approx 100\%$) and test accuracy.

### The Solution: Regularisation via Hyperparameters
We constrain tree complexity using three regularisation levers:

| Hyperparameter | Effect | Range |
|---|---|---|
| `max_depth` | Limits the maximum number of sequential splits from root to leaf. Prevents deep, overly specific branches. | `[5, 10, 15, 20, None]` |
| `min_samples_split` | Minimum number of training samples required to split an internal node. Higher values create more conservative (generalised) splits. | `[2, 5, 10, 20]` |
| `min_samples_leaf` | Minimum number of samples required at each leaf node. Prevents leaves that represent a single noisy training point. | `[1, 2, 5]` |
| `criterion` | The impurity measure used to evaluate split quality. | `['gini', 'entropy']` |

### Method: 5-Fold Stratified Cross-Validation
`GridSearchCV` exhaustively evaluates every combination in the hyperparameter grid using **5-fold stratified cross-validation** (StratifiedKFold):

$$\text{CV Score} = \frac{1}{K} \sum_{k=1}^{K} \text{F1}(f_{\theta}^{(-k)}, D^{(k)})$$

where $f_{\theta}^{(-k)}$ is the model trained on 4 folds (excluding fold $k$), and $D^{(k)}$ is the held-out validation fold. Stratification ensures each fold preserves the class ratio.

**Total configurations**: $5 \times 4 \times 3 \times 2 = 120$ combinations $\times$ 5 folds = **600 model fits**.


In [ ]:
# =====================================================================
# STEP 10.1: Define Hyperparameter Grid & Run GridSearchCV
# =====================================================================
param_grid = {
    'max_depth':        [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 5],
    'criterion':         ['gini', 'entropy']
}

total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f"Hyperparameter grid: {total_combos} combinations × 5 folds = {total_combos * 5} model fits")
print(f"Grid: {param_grid}")

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("\nStarting GridSearchCV (this may take a few minutes)...")
grid_search.fit(X_train_sel, y_train)

print("\n" + "=" * 70)
print("  GRIDSEARCHCV COMPLETE")
print("=" * 70)
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")
print(f"Best Parameters: {grid_search.best_params_}")


In [ ]:
# =====================================================================
    # STEP 10.2: GridSearchCV Results Analysis
# =====================================================================
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

# Display top 10 configurations
display_cols = ['rank_test_score', 'mean_test_score', 'std_test_score',
                'mean_train_score', 'param_criterion', 'param_max_depth',
                'param_min_samples_split', 'param_min_samples_leaf']
print("=== Top 10 GridSearchCV Configurations ===")
print(results_df[display_cols].head(10).to_string(index=False))

# Compute overfitting gap for top configs
results_df['overfit_gap'] = results_df['mean_train_score'] - results_df['mean_test_score']
print(f"\nTop config overfitting gap (train - val): {results_df.iloc[0]['overfit_gap']:.4f}")


In [ ]:
# =====================================================================
# STEP 10.3: Visualize GridSearchCV Results — max_depth vs F1
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: max_depth effect (averaged over other params)
for criterion in ['gini', 'entropy']:
    mask = results_df['param_criterion'] == criterion
    grouped = results_df[mask].groupby('param_max_depth').agg(
        mean_val=('mean_test_score', 'mean'),
        mean_train=('mean_train_score', 'mean')
    ).reset_index()

    depths_plot = [str(d) for d in grouped['param_max_depth']]
    axes[0].plot(depths_plot, grouped['mean_val'], '-o', label=f'{criterion} (Val)', markersize=5)
    axes[0].plot(depths_plot, grouped['mean_train'], '--s', label=f'{criterion} (Train)', markersize=3, alpha=0.5)

axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Mean F1 Score')
axes[0].set_title('Effect of max_depth on F1 Score', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: min_samples_split effect
for criterion in ['gini', 'entropy']:
    mask = results_df['param_criterion'] == criterion
    grouped = results_df[mask].groupby('param_min_samples_split').agg(
        mean_val=('mean_test_score', 'mean'),
    ).reset_index()
    axes[1].plot(grouped['param_min_samples_split'], grouped['mean_val'], '-o',
                 label=f'{criterion}', markersize=5)

axes[1].set_xlabel('min_samples_split')
axes[1].set_ylabel('Mean CV F1 Score')
axes[1].set_title('Effect of min_samples_split on F1 Score', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Stage 10: GridSearchCV — Hyperparameter Effects on Model Performance',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage10_gridsearch_hyperparameter_effects.png', dpi=300)
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage10_gridsearch_hyperparameter_effects.png'}")


---
## Final Tuned Model: Evaluation on Pristine Test Set

We now evaluate the **best model** identified by GridSearchCV on the **pristine, untouched test set** ($X_{\text{test}}, y_{\text{test}}$) that reflects the authentic real-world class distribution (0.25% attack rate).

This is the definitive performance assessment — no SMOTE, no cross-validation, no data leakage.


In [ ]:
# =====================================================================
# FINAL MODEL: Best Tuned Decision Tree
# =====================================================================
dt_tuned = grid_search.best_estimator_

y_pred_tuned_train = dt_tuned.predict(X_train_sel)
y_pred_tuned_test  = dt_tuned.predict(X_test_sel)

print("=" * 70)
print("  FINAL TUNED CART DECISION TREE — TEST SET EVALUATION")
print("=" * 70)
print(f"\nBest Hyperparameters: {grid_search.best_params_}")
print(f"Tree Depth: {dt_tuned.get_depth()}")
print(f"Number of Leaves: {dt_tuned.get_n_leaves()}")

print("\n--- Training Set Performance ---")
print(f"Accuracy:  {accuracy_score(y_train, y_pred_tuned_train):.4f}")
print(f"Precision: {precision_score(y_train, y_pred_tuned_train):.4f}")
print(f"Recall:    {recall_score(y_train, y_pred_tuned_train):.4f}")
print(f"F1-Score:  {f1_score(y_train, y_pred_tuned_train):.4f}")

print("\n--- Test Set Performance (DEFINITIVE) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_tuned_test):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_tuned_test):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_tuned_test):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_tuned_test):.4f}")

print("\n--- Full Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_tuned_test, target_names=['Benign (0)', 'Malicious (1)']))


In [ ]:
# =====================================================================
# FINAL: Confusion Matrix — Baseline vs Tuned Comparison
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline (unconstrained) test confusion matrix
cm_baseline = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(cm_baseline, display_labels=['Benign', 'Attack']).plot(
    ax=axes[0], cmap='Oranges', values_format=',d')
axes[0].set_title(f'Baseline (depth={dt_baseline.get_depth()})\nTest F1={f1_score(y_test, y_pred_test):.4f}',
                  fontsize=12, fontweight='bold')

# Tuned test confusion matrix
cm_tuned = confusion_matrix(y_test, y_pred_tuned_test)
ConfusionMatrixDisplay(cm_tuned, display_labels=['Benign', 'Attack']).plot(
    ax=axes[1], cmap='Greens', values_format=',d')
axes[1].set_title(f'Tuned (depth={dt_tuned.get_depth()}, GridSearchCV)\nTest F1={f1_score(y_test, y_pred_tuned_test):.4f}',
                  fontsize=12, fontweight='bold')

plt.suptitle('Stage 10: Baseline vs Tuned CART — Test Set Confusion Matrices',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage10_baseline_vs_tuned_confusion.png', dpi=300)
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage10_baseline_vs_tuned_confusion.png'}")


In [ ]:
# =====================================================================
# FINAL: Tuned Decision Tree Visualization
# =====================================================================
plt.figure(figsize=(28, 14))
plot_tree(
    dt_tuned,
    max_depth=4,
    feature_names=X_train_sel.columns.tolist(),
    class_names=['Benign', 'Attack'],
    filled=True,
    rounded=True,
    fontsize=8,
    proportion=True,
    impurity=True
)
plt.title('Stage 10: Tuned CART Decision Tree — Top 4 Levels (Optimised Rule Diagram)',
          fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage10_tuned_decision_tree.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage10_tuned_decision_tree.png'}")


In [ ]:
# =====================================================================
# FINAL: Tuned Model Feature Importance
# =====================================================================
tuned_importances = dt_tuned.feature_importances_
tuned_feat_imp = pd.DataFrame({
    'Feature': X_train_sel.columns,
    'Gini_Importance': tuned_importances
}).sort_values('Gini_Importance', ascending=False)

plt.figure(figsize=(10, 6))
top_n = min(15, len(tuned_feat_imp))
top_feats_tuned = tuned_feat_imp.head(top_n)
colors = ['#d62728' if f in ['entropy_x_length', 'tunneling_rr_score', 'network_diversity']
          else '#2ca02c' for f in top_feats_tuned['Feature']]
plt.barh(top_feats_tuned['Feature'], top_feats_tuned['Gini_Importance'], color=colors)
plt.xlabel('Gini Feature Importance')
plt.title('Stage 10: Tuned CART — Feature Importances (Top 15)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage10_tuned_feature_importance.png', dpi=300)
plt.show()
print(f"Saved: {FIGURES_DIR / 'stage10_tuned_feature_importance.png'}")

print("\nFull Tuned Feature Importance Ranking:")
print(tuned_feat_imp.to_string(index=False))


---
## Comprehensive Performance Summary

### Baseline vs Tuned CART Comparison


In [ ]:
# =====================================================================
# Performance Summary Table
# =====================================================================
summary_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Tree Depth', 'Leaves', 'CV F1 (5-fold)'],
    'Baseline (Train)': [
        f"{accuracy_score(y_train, y_pred_train):.4f}",
        f"{precision_score(y_train, y_pred_train):.4f}",
        f"{recall_score(y_train, y_pred_train):.4f}",
        f"{f1_score(y_train, y_pred_train):.4f}",
        str(dt_baseline.get_depth()),
        str(dt_baseline.get_n_leaves()),
        "N/A"
    ],
    'Baseline (Test)': [
        f"{accuracy_score(y_test, y_pred_test):.4f}",
        f"{precision_score(y_test, y_pred_test):.4f}",
        f"{recall_score(y_test, y_pred_test):.4f}",
        f"{f1_score(y_test, y_pred_test):.4f}",
        str(dt_baseline.get_depth()),
        str(dt_baseline.get_n_leaves()),
        "N/A"
    ],
    'Tuned (Train)': [
        f"{accuracy_score(y_train, y_pred_tuned_train):.4f}",
        f"{precision_score(y_train, y_pred_tuned_train):.4f}",
        f"{recall_score(y_train, y_pred_tuned_train):.4f}",
        f"{f1_score(y_train, y_pred_tuned_train):.4f}",
        str(dt_tuned.get_depth()),
        str(dt_tuned.get_n_leaves()),
        f"{grid_search.best_score_:.4f}"
    ],
    'Tuned (Test)': [
        f"{accuracy_score(y_test, y_pred_tuned_test):.4f}",
        f"{precision_score(y_test, y_pred_tuned_test):.4f}",
        f"{recall_score(y_test, y_pred_tuned_test):.4f}",
        f"{f1_score(y_test, y_pred_tuned_test):.4f}",
        str(dt_tuned.get_depth()),
        str(dt_tuned.get_n_leaves()),
        f"{grid_search.best_score_:.4f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("=" * 90)
print("  COMPREHENSIVE PERFORMANCE COMPARISON: BASELINE vs TUNED CART")
print("=" * 90)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv(PROCESSED_DIR / 'stage_7_8_10_performance_summary.csv', index=False)
print(f"\nSaved: {PROCESSED_DIR / 'stage_7_8_10_performance_summary.csv'}")


---
## Deliverables Export & Handoff to Jaimin

All processed feature matrices, trained models, and evaluation artifacts are now exported for downstream use.


In [ ]:
# =====================================================================
# EXPORT: Save engineered feature matrices and tuned model
# =====================================================================

# Save engineered + selected feature matrices
X_train_sel.to_csv(PROCESSED_DIR / 'X_train_engineered.csv', index=False)
X_test_sel.to_csv(PROCESSED_DIR / 'X_test_engineered.csv', index=False)

# Save tuned model
joblib.dump(dt_tuned, PROCESSED_DIR / 'cart_tuned_model.joblib')
joblib.dump(dt_baseline, PROCESSED_DIR / 'cart_baseline_model.joblib')

# Save GridSearchCV results
results_df.to_csv(PROCESSED_DIR / 'gridsearch_cv_results.csv', index=False)

# Save best hyperparameters
import json
with open(PROCESSED_DIR / 'best_hyperparameters.json', 'w') as f:
    json.dump(grid_search.best_params_, f, indent=2, default=str)

print("=" * 70)
print("  STAGE 7, 8 & 10 (Part 1) — ALL DELIVERABLES EXPORTED")
print("=" * 70)
print(f"\nEngineered Features:")
print(f"  X_train_engineered.csv: {X_train_sel.shape}")
print(f"  X_test_engineered.csv:  {X_test_sel.shape}")
print(f"\nModels:")
print(f"  cart_baseline_model.joblib  (depth={dt_baseline.get_depth()}, leaves={dt_baseline.get_n_leaves()})")
print(f"  cart_tuned_model.joblib     (depth={dt_tuned.get_depth()}, leaves={dt_tuned.get_n_leaves()})")
print(f"\nTuning:")
print(f"  gridsearch_cv_results.csv   ({len(results_df)} configurations)")
print(f"  best_hyperparameters.json   ({grid_search.best_params_})")
print(f"\nFigures:")
for fig_file in sorted(FIGURES_DIR.glob('stage*')):
    print(f"  {fig_file}")
print(f"\nHandoff Notes for Jaimin:")
print(f"  - Use X_train_engineered.csv and X_test_engineered.csv as feature inputs")
print(f"  - Baseline CART test F1: {f1_score(y_test, y_pred_tuned_test):.4f}")
print(f"  - Neural Networks / SVMs must EXCEED this interpretable baseline to justify complexity")
print("=" * 70)
